# Semana 01 · Ciclo de vida completo de un modelo con MLflow

En esta práctica llevarás un caso de clasificación desde los datos hasta una API local:

`datos → experimentos → comparación → gate → modelo ganador → Registry → API → observabilidad`

El objetivo no es conseguir el número más alto, sino construir una decisión **reproducible,
auditable y desplegable**. Usaremos Wine Quality como caso didáctico de clasificación
multiclase: predeciremos la calidad original y no debe interpretarse como garantía comercial.

**Entorno:** Databricks Free Edition, compute serverless y experimento asociado a este notebook.
No necesitas tarjeta de crédito, cluster propio, Docker, `uv`, `pyproject.toml` ni tokens.

## Mapa y criterios de terminación

Al acabar debes poder enseñar en MLflow:

1. un lote de al menos siete runs comparables, con datos, parámetros, métricas y artefactos;
2. una regla de selección escrita **antes** de mirar el test;
3. un único candidato evaluado sobre test y marcado como ganador;
4. versiones en Unity Catalog y un ciclo `Challenger → Champion → rollback`;
5. una API HTTP local que carga el `model.joblib` del run ganador y supera pruebas válidas e inválidas;
6. un run de despliegue con latencia, volumen de peticiones, errores y evidencia de la respuesta.

Tiempo recomendado: 45 min de construcción guiada + 120–180 min de práctica autónoma y debrief.

## 0. Dependencias

Ejecuta `%pip` al principio. Si Databricks reinicia Python, continúa desde la celda de imports.
Las dependencias se instalan en la sesión del notebook; no se crea ningún entorno o fichero de
proyecto en el repositorio.

In [0]:
%pip install -q --upgrade "mlflow[databricks]==3.14.0" "scikit-learn==1.9.0" "pandas==2.2.3" "matplotlib==3.11.1" "joblib==1.5.3" "xgboost==3.0.5"

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

## Cómo trabajar con la versión sin resolver

Ejecuta las celdas en orden y completa cada `TODO`. Los bloques se detienen con
`NotImplementedError` para que un fallo no genere evidencia engañosa. Consulta la guía del
alumno antes de mirar la solución. Mantén el test cerrado hasta haber elegido el ganador.

## 1. Configuración e identidad del lote

In [0]:
import json
import joblib
import re
import tempfile
import threading
import time
import uuid
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import sklearn
import xgboost
from mlflow import MlflowClient
from mlflow.models import infer_signature
from sklearn.ensemble import ExtraTreesClassifier
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, classification_report,
    f1_score, precision_score, recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

STUDENT_ALIAS = "202114423"
BATCH_ID = uuid.uuid4().hex[:8]
RANDOM_STATE = 42
TRAIN_SIZE = 0.70
VALIDATION_SIZE = 0.15
TEST_SIZE = 0.15
assert abs(TRAIN_SIZE + VALIDATION_SIZE + TEST_SIZE - 1.0) < 1e-9
MIN_VALIDATION_F1_MACRO = 0.28
QUALITY_MIN = 3
QUALITY_MAX = 8
NUM_CLASSES = QUALITY_MAX - QUALITY_MIN + 1
# Esta ruta es relativa a semana1/modules/01-mlflow-databricks-foundations/notebooks/.
DATASET_PATH = "../../../data/raw/WineQT.csv"

# --- TODO 1: validación del alias y configuración de tracking ---
assert re.fullmatch(r"\d{7,9}", STUDENT_ALIAS), f"Alias '{STUDENT_ALIAS}' debe ser 7-9 dígitos"

mlflow.set_tracking_uri("databricks")
EXPERIMENT_NAME = f"/Users/pedroperezdiaz1@gmail.com/mlops_wine_quality_s01"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Alias: {STUDENT_ALIAS} | Batch: {BATCH_ID}")
print(f"Experiment: {EXPERIMENT_NAME}")

Alias: 202114423 | Batch: 8b01ae64
Experiment: /Users/pedroperezdiaz1@gmail.com/mlops_wine_quality_s01


If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


## 2. Contrato y controles de datos

Permite nulos porque el pipeline los imputará, pero detente si faltan columnas, el dataset está
vacío, falta el target o `Id` se cuela como predictor. Construye un `quality_report`
serializable como JSON y confirma que el target contiene al menos dos clases.

In [0]:
data = pd.read_csv(DATASET_PATH)
TARGET = "quality"
EXPECTED_FEATURES = [
    "fixed acidity", "volatile acidity", "citric acid", "residual sugar",
    "chlorides", "free sulfur dioxide", "total sulfur dioxide", "density",
    "pH", "sulphates", "alcohol",
]

# --- TODO 2: validación del contrato y creación de X, y, quality_report ---
if data.empty:
    raise ValueError("El dataset está vacío")
if TARGET not in data.columns:
    raise ValueError(f"Falta la columna target '{TARGET}'")
missing_features = [f for f in EXPECTED_FEATURES if f not in data.columns]
if missing_features:
    raise ValueError(f"Faltan columnas: {missing_features}")
if "Id" in EXPECTED_FEATURES:
    raise ValueError("Id no debe ser predictor")

unique_qualities = sorted(data[TARGET].unique())
qualities_in_range = [q for q in unique_qualities if QUALITY_MIN <= q <= QUALITY_MAX]
qualities_out_of_range = [q for q in unique_qualities if q < QUALITY_MIN or q > QUALITY_MAX]
if qualities_out_of_range:
    raise ValueError(f"Calidades fuera de rango [{QUALITY_MIN}, {QUALITY_MAX}]: {qualities_out_of_range}")

X = data[EXPECTED_FEATURES].copy()
y = (data[TARGET] - QUALITY_MIN).copy()
y.name = "quality_class"

quality_report = {
    "rows": int(len(data)),
    "duplicates": int(data.duplicated().sum()),
    "original_quality_distribution": {str(k): int(v) for k, v in data[TARGET].value_counts().sort_index().items()},
    "encoded_quality_distribution": {str(k): int(v) for k, v in y.value_counts().sort_index().items()},
    "class_mapping": {str(q): int(q - QUALITY_MIN) for q in qualities_in_range},
    "num_classes": len(qualities_in_range),
    "nulls_per_column": {col: int(data[col].isnull().sum()) for col in EXPECTED_FEATURES},
    "checks": {
        "has_target": True,
        "has_all_features": len(missing_features) == 0,
        "id_excluded": "Id" not in EXPECTED_FEATURES,
        "at_least_two_classes": y.nunique() >= 2,
        "qualities_in_range": len(qualities_out_of_range) == 0,
    },
}
assert y.nunique() >= 2, "El target debe tener al menos 2 clases"
print(f"X shape: {X.shape} | y classes: {sorted(y.unique())}")
print(f"Quality report: {json.dumps(quality_report, indent=2)}")

X shape: (1143, 11) | y classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]
Quality report: {
  "rows": 1143,
  "duplicates": 0,
  "original_quality_distribution": {
    "3": 6,
    "4": 33,
    "5": 483,
    "6": 462,
    "7": 143,
    "8": 16
  },
  "encoded_quality_distribution": {
    "0": 6,
    "1": 33,
    "2": 483,
    "3": 462,
    "4": 143,
    "5": 16
  },
  "class_mapping": {
    "3": 0,
    "4": 1,
    "5": 2,
    "6": 3,
    "7": 4,
    "8": 5
  },
  "num_classes": 6,
  "nulls_per_column": {
    "fixed acidity": 0,
    "volatile acidity": 0,
    "citric acid": 0,
    "residual sugar": 0,
    "chlorides": 0,
    "free sulfur dioxide": 0,
    "total sulfur dioxide": 0,
    "density": 0,
    "pH": 0,
    "sulphates": 0,
    "alcohol": 0
  },
  "checks": {
    "has_target": true,
    "has_all_features": true,
    "id_excluded": true,
    "at_least_two_classes": true,
    "qualities_in_range": true
  }
}


## 3. Divide 70 % train, 15 % validación y 15 % test, siempre estratificado

In [0]:
# --- TODO 3: división estratificada 70/15/15 ---
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
valid_frac = VALIDATION_SIZE / (TRAIN_SIZE + VALIDATION_SIZE)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp, y_temp, test_size=valid_frac, random_state=RANDOM_STATE, stratify=y_temp
)
print(f"Train: {X_train.shape[0]} | Valid: {X_valid.shape[0]} | Test: {X_test.shape[0]}")

Train: 799 | Valid: 172 | Test: 172


## 4. Pipeline y funciones de evaluación

In [0]:
def build_pipeline(config: dict) -> Pipeline:
    cfg = dict(config)
    cfg.pop("candidate_name", "unnamed")
    model_type = cfg.pop("model_type", "extratrees")
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if model_type == "extratrees":
        steps.append(("classifier", ExtraTreesClassifier(**cfg)))
    elif model_type == "xgboost":
        steps.append(("classifier", XGBClassifier(**cfg)))
    else:
        raise ValueError(f"model_type desconocido: {model_type}")
    return Pipeline(steps)


def evaluate_classifier(model: Pipeline, features: pd.DataFrame, labels: pd.Series) -> dict:
    start = time.perf_counter()
    predictions = model.predict(features)
    elapsed_ms = (time.perf_counter() - start) * 1000
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "precision_macro": float(precision_score(labels, predictions, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(labels, predictions, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(labels, predictions, average="macro", zero_division=0)),
        "f1_weighted": float(f1_score(labels, predictions, average="weighted", zero_division=0)),
        "latency_ms_per_row": float(elapsed_ms / len(features)),
    }

## 5. Diseña al menos siete candidatos comparables

In [0]:
# --- TODO 5: al menos siete candidatos comparables ---
CANDIDATES = [
    {
        "candidate_name": "et_300_none",
        "model_type": "extratrees",
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 1,
        "max_features": 1.0,
        "class_weight": None,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    },
    {
        "candidate_name": "et_200_10",
        "model_type": "extratrees",
        "n_estimators": 200,
        "max_depth": 10,
        "min_samples_leaf": 2,
        "max_features": 1.0,
        "class_weight": None,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    },
    {
        "candidate_name": "et_500_balanced",
        "model_type": "extratrees",
        "n_estimators": 500,
        "max_depth": None,
        "min_samples_leaf": 1,
        "max_features": "sqrt",
        "class_weight": "balanced",
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    },
    {
        "candidate_name": "et_100_5",
        "model_type": "extratrees",
        "n_estimators": 100,
        "max_depth": 5,
        "min_samples_leaf": 4,
        "max_features": 0.5,
        "class_weight": None,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    },
    {
        "candidate_name": "et_300_leaf3",
        "model_type": "extratrees",
        "n_estimators": 300,
        "max_depth": 15,
        "min_samples_leaf": 3,
        "max_features": 1.0,
        "class_weight": None,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    },
    {
        "candidate_name": "xgb_lr01_md5",
        "model_type": "xgboost",
        "objective": "multi:softprob",
        "num_class": NUM_CLASSES,
        "eval_metric": "mlogloss",
        "learning_rate": 0.1,
        "max_depth": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "n_estimators": 200,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    },
    {
        "candidate_name": "xgb_lr005_md7",
        "model_type": "xgboost",
        "objective": "multi:softprob",
        "num_class": NUM_CLASSES,
        "eval_metric": "mlogloss",
        "learning_rate": 0.05,
        "max_depth": 7,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "n_estimators": 300,
        "random_state": RANDOM_STATE,
        "n_jobs": -1,
    },
]
assert len(CANDIDATES) >= 7, "Necesitas al menos siete candidatos, incluyendo XGBoost."
assert any(candidate.get("model_type") == "xgboost" for candidate in CANDIDATES)
print(f"{len(CANDIDATES)} candidatos definidos")

7 candidatos definidos


## 6. Tracking profundo: un run por candidato

Por cada candidato registra tags, parámetros, versiones, inputs de train/validación, tarjeta de
datos, calidad, riesgos, métricas, reporte por clase, matriz de confusión, modelo MLflow con
firma/input example, `deployment/model.joblib` y contrato de inferencia.

In [0]:
dataset_card = {
    "source": "WineQT.csv — UCI Wine Quality (red variants), Kaggle mirror",
    "purpose": "Clasificación multiclase didáctica de calidad de vino (3–8) para práctica del ciclo de vida MLOps",
    "features": EXPECTED_FEATURES,
    "target": "quality_class",
    "target_definition": f"quality original 3..8; quality_class = quality - {QUALITY_MIN}",
    "split": {"train": 0.70, "validation": 0.15, "test": 0.15, "random_state": RANDOM_STATE},
    "known_limitations": [
        "Clases extremas (3 y 8) son muy raras, causando desbalance severo",
        "Dataset pequeño (~1k filas), limitando generalización para clases minoritarias",
        "Sin metadatos temporales ni de productor; los resultados no son garantías comerciales",
    ],
}
risk_register = {
    "version": "s01-lifecycle",
    "risks": [
        {"id": "R1", "risk": "Desbalance de clases en calidades extremas (3, 8)", "impact": "high", "owner": STUDENT_ALIAS, "mitigation": "Usar F1 macro como métrica del gate; considerar class_weight o sampling"},
        {"id": "R2", "risk": "Fuga de datos desde columna Id o features derivados del target", "impact": "high", "owner": STUDENT_ALIAS, "mitigation": "Excluir Id de predictores; validar lista de features explícitamente"},
        {"id": "R3", "risk": "Overfitting por dataset pequeño y múltiples candidatos", "impact": "high", "owner": STUDENT_ALIAS, "mitigation": "Reservar test hasta evaluación final; gate sobre validación antes de test"},
        {"id": "R4", "risk": "Serving skew entre pipeline de entrenamiento y API de inferencia", "impact": "high", "owner": STUDENT_ALIAS, "mitigation": "Imputer + modelo en un solo Pipeline; serializar como joblib para la API"},
    ],
}
assert len(dataset_card["known_limitations"]) >= 3
assert len(risk_register["risks"]) >= 4
print("Dataset card y risk register completados")

Dataset card y risk register completados


### 6a. Inputs de datos

Crea dos objetos `mlflow.data` con nombres distintos. Comprueba sus nombres antes de lanzar runs.

In [0]:
# --- TODO 6a: datasets de MLflow ---
import mlflow.data

train_df = X_train.copy()
train_df["quality_class"] = y_train

validation_df = X_valid.copy()
validation_df["quality_class"] = y_valid

train_dataset = mlflow.data.from_pandas(
    train_df, source=DATASET_PATH, name="wine_train", targets="quality_class"
)
validation_dataset = mlflow.data.from_pandas(
    validation_df, source=DATASET_PATH, name="wine_validation", targets="quality_class"
)
print(f"Train dataset: {train_dataset.name} | {train_df.shape}")
print(f"Validation dataset: {validation_dataset.name} | {validation_df.shape}")

Train dataset: wine_train | (799, 12)
Validation dataset: wine_validation | (172, 12)


/local_disk0/.ephemeral_nfs/envs/pythonEnv-cf367e1e-9f5d-4716-9b4b-37fcbe89f8d7/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-cf367e1e-9f5d-4716-9b4b-37fcbe89f8d7/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


### 6b. Artefactos del candidato

Implementa una función pequeña que registre firma/input example, `model_info.model_uri`, Joblib,
tamaño, matriz de confusión y contrato. Podrás probarla dentro de un único run antes del bucle.

In [0]:
def log_candidate_artifacts(
    model: Pipeline, predictions: np.ndarray, candidate_index: int
) -> str:
    input_example = X_train.head(3)
    signature = infer_signature(input_example, model.predict(input_example))

    # Collect trusted types for skops serialization
    trusted_types = ["numpy.dtype"]
    if any(isinstance(step[1], XGBClassifier) for step in model.steps):
        trusted_types.extend([
            "xgboost.sklearn.XGBClassifier",
            "xgboost.core.Booster",
            "xgboost.DMatrix",
        ])

    try:
        model_info = mlflow.sklearn.log_model(
            model, name="model", signature=signature,
            input_example=input_example, skops_trusted_types=trusted_types,
        )
    except TypeError:
        model_info = mlflow.sklearn.log_model(
            model, name="model", signature=signature,
            input_example=input_example,
        )

    model_uri = model_info.model_uri
    mlflow.set_tag("candidate.model_uri", model_uri)

    with tempfile.TemporaryDirectory() as tmpdir:
        # Joblib serialization
        joblib_path = Path(tmpdir) / "model.joblib"
        joblib.dump(model, joblib_path)
        joblib_size = joblib_path.stat().st_size
        mlflow.log_artifact(str(joblib_path), artifact_path="deployment")
        mlflow.log_metric("joblib_size_bytes", float(joblib_size))

        # Confusion matrix
        labels_sorted = sorted(y_valid.unique())
        cm = ConfusionMatrixDisplay.from_predictions(
            y_valid, predictions, display_labels=labels_sorted, normalize=None
        )
        cm_path = Path(tmpdir) / "confusion_matrix.png"
        cm.figure_.savefig(cm_path, dpi=100, bbox_inches="tight")
        mlflow.log_artifact(str(cm_path), artifact_path="artifacts")
        plt.close(cm.figure_)

        # Inference contract
        inference_contract = {
            "input_schema": {
                "type": "dataframe",
                "columns": EXPECTED_FEATURES,
                "dtypes": {col: "float64" for col in EXPECTED_FEATURES},
            },
            "output_schema": {
                "type": "array",
                "description": f"Integer class labels 0..{NUM_CLASSES - 1} (quality_class = quality - {QUALITY_MIN})",
            },
            "model_uri": model_uri,
            "candidate_index": candidate_index,
        }
        contract_path = Path(tmpdir) / "inference_contract.json"
        contract_path.write_text(json.dumps(inference_contract, indent=2))
        mlflow.log_artifact(str(contract_path), artifact_path="deployment")

    return model_uri

### 6c. Runs comparables

Ahora el bucle se ocupa sólo de identidad, parámetros, inputs, gobierno, entrenamiento, métricas
y de llamar al helper de artefactos. Conserva `experiment_id` y cada `run_id`.

In [0]:
experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
candidate_run_ids = []

for i, candidate in enumerate(CANDIDATES):
    candidate_name = candidate["candidate_name"]
    model_type = candidate.get("model_type", "extratrees")

    with mlflow.start_run(experiment_id=experiment_id, run_name=f"{candidate_name}_{BATCH_ID}") as run:
        run_id = run.info.run_id
        candidate_run_ids.append(run_id)

        # Tags
        mlflow.set_tag("student_alias", STUDENT_ALIAS)
        mlflow.set_tag("batch_id", BATCH_ID)
        mlflow.set_tag("candidate_name", candidate_name)
        mlflow.set_tag("model_type", model_type)
        mlflow.set_tag("sklearn_version", sklearn.__version__)
        if model_type == "xgboost":
            mlflow.set_tag("xgboost_version", xgboost.__version__)

        # Parameters
        for key, value in candidate.items():
            if key not in ("candidate_name", "model_type"):
                mlflow.log_param(key, value)

        # Datasets
        mlflow.log_input(train_dataset, context="training")
        mlflow.log_input(validation_dataset, context="validation")

        # Governance artifacts
        mlflow.log_dict(dataset_card, "dataset_card.json")
        mlflow.log_dict(risk_register, "risk_register.json")
        mlflow.log_dict(quality_report, "quality_report.json")

        # Build and train
        pipeline = build_pipeline(candidate)
        pipeline.fit(X_train, y_train)

        # Evaluate on validation
        val_metrics = evaluate_classifier(pipeline, X_valid, y_valid)
        predictions = pipeline.predict(X_valid)

        # Log metrics
        for key, value in val_metrics.items():
            mlflow.log_metric(f"validation.{key}", value)

        # Per-class metrics
        report = classification_report(y_valid, predictions, output_dict=True, zero_division=0)
        mlflow.log_dict(report, "validation_classification_report.json")
        for cls in range(NUM_CLASSES):
            cls_str = str(cls)
            if cls_str in report:
                mlflow.log_metric(f"validation.f1_class_{cls}", report[cls_str]["f1-score"])

        # Artifacts
        model_uri = log_candidate_artifacts(pipeline, predictions, i)

        # Print summary row
        print(f"[{i + 1}/{len(CANDIDATES)}] {candidate_name} | "
              f"F1_macro={val_metrics['f1_macro']:.4f} | "
              f"Acc={val_metrics['accuracy']:.4f} | "
              f"Latency={val_metrics['latency_ms_per_row']:.4f}ms | "
              f"run_id={run_id}")

assert len(candidate_run_ids) == len(CANDIDATES)

2026/10/01 10:53:52 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: An error occurred while calling o12.extraContext. Trace:
py4j.security.Py4JSecurityException: Method public scala.collection.immutable.Map com.databricks.backend.common.rpc.CommandContext.extraContext() is not whitelisted on class class com.databricks.backend.common.rpc.CommandContext
	at py4j.security.WhitelistingPy4JSecurityManager.checkCall(WhitelistingPy4JSecurityManager.java:473)
	at py4j.Gateway.invoke(Gateway.java:305)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.$anonfun$invokeMethod$1(InstrumentedCallCommand.scala:19)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke(InstrumentedPy4jCommand.scala:28)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke$(InstrumentedPy4jCommand.scala:18)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.instrumentedInvoke(Instrumented

[1/7] et_300_none | F1_macro=0.2957 | Acc=0.6337 | Latency=0.4026ms | run_id=8954b9669eda4cf6bfbb10ab396c050f


2026/10/01 10:54:23 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: An error occurred while calling o127.extraContext. Trace:
py4j.security.Py4JSecurityException: Method public scala.collection.immutable.Map com.databricks.backend.common.rpc.CommandContext.extraContext() is not whitelisted on class class com.databricks.backend.common.rpc.CommandContext
	at py4j.security.WhitelistingPy4JSecurityManager.checkCall(WhitelistingPy4JSecurityManager.java:473)
	at py4j.Gateway.invoke(Gateway.java:305)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.$anonfun$invokeMethod$1(InstrumentedCallCommand.scala:19)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke(InstrumentedPy4jCommand.scala:28)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke$(InstrumentedPy4jCommand.scala:18)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.instrumentedInvoke(Instrumente

[2/7] et_200_10 | F1_macro=0.2955 | Acc=0.6453 | Latency=0.3316ms | run_id=014cfea50107424297c50b585f7b7ccb


2026/10/01 10:54:42 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: An error occurred while calling o197.extraContext. Trace:
py4j.security.Py4JSecurityException: Method public scala.collection.immutable.Map com.databricks.backend.common.rpc.CommandContext.extraContext() is not whitelisted on class class com.databricks.backend.common.rpc.CommandContext
	at py4j.security.WhitelistingPy4JSecurityManager.checkCall(WhitelistingPy4JSecurityManager.java:473)
	at py4j.Gateway.invoke(Gateway.java:305)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.$anonfun$invokeMethod$1(InstrumentedCallCommand.scala:19)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke(InstrumentedPy4jCommand.scala:28)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke$(InstrumentedPy4jCommand.scala:18)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.instrumentedInvoke(Instrumente

...

[5/7] et_300_leaf3 | F1_macro=0.2978 | Acc=0.6512 | Latency=0.3793ms | run_id=4a7f69f8081d48e69854fccbc2babdbe


2026/10/01 10:55:37 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: An error occurred while calling o410.extraContext. Trace:
py4j.security.Py4JSecurityException: Method public scala.collection.immutable.Map com.databricks.backend.common.rpc.CommandContext.extraContext() is not whitelisted on class class com.databricks.backend.common.rpc.CommandContext
	at py4j.security.WhitelistingPy4JSecurityManager.checkCall(WhitelistingPy4JSecurityManager.java:473)
	at py4j.Gateway.invoke(Gateway.java:305)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.$anonfun$invokeMethod$1(InstrumentedCallCommand.scala:19)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke(InstrumentedPy4jCommand.scala:28)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke$(InstrumentedPy4jCommand.scala:18)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.instrumentedInvoke(Instrumente

[6/7] xgb_lr01_md5 | F1_macro=0.2755 | Acc=0.5872 | Latency=0.0220ms | run_id=fac01704afb344b9885faa781262cc9f


2026/10/01 10:55:57 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: An error occurred while calling o484.extraContext. Trace:
py4j.security.Py4JSecurityException: Method public scala.collection.immutable.Map com.databricks.backend.common.rpc.CommandContext.extraContext() is not whitelisted on class class com.databricks.backend.common.rpc.CommandContext
	at py4j.security.WhitelistingPy4JSecurityManager.checkCall(WhitelistingPy4JSecurityManager.java:473)
	at py4j.Gateway.invoke(Gateway.java:305)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.$anonfun$invokeMethod$1(InstrumentedCallCommand.scala:19)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke(InstrumentedPy4jCommand.scala:28)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke$(InstrumentedPy4jCommand.scala:18)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.instrumentedInvoke(Instrumente

[7/7] xgb_lr005_md7 | F1_macro=0.2794 | Acc=0.5930 | Latency=0.0292ms | run_id=cdcd6622feee4a58bbc3ce15206fdb52


## 7. Busca y compara exclusivamente los runs de tu alias y `BATCH_ID`

In [0]:
# --- TODO 7: comparación de runs ---
filter_string = (
    f"tags.student_alias = '{STUDENT_ALIAS}' "
    f"and tags.batch_id = '{BATCH_ID}' "
    f"and tags.candidate_name != ''"
)
comparison = mlflow.search_runs(
    experiment_ids=[experiment_id],
    filter_string=filter_string,
    order_by=["metrics.validation.f1_macro DESC"],
)
comparison = comparison[comparison["tags.candidate_name"].notna()]

display_cols = [
    "tags.candidate_name", "tags.model_type",
    "metrics.validation.f1_macro", "metrics.validation.accuracy",
    "metrics.validation.precision_macro", "metrics.validation.recall_macro",
    "metrics.validation.f1_weighted", "metrics.validation.latency_ms_per_row",
    "run_id",
]
comparison_display = comparison[[c for c in display_cols if c in comparison.columns]]
display(comparison_display)

tags.candidate_name,tags.model_type,metrics.validation.f1_macro,metrics.validation.accuracy,metrics.validation.precision_macro,metrics.validation.recall_macro,metrics.validation.f1_weighted,metrics.validation.latency_ms_per_row,run_id
et_300_leaf3,extratrees,0.2978088511350025,0.6511627906976745,0.30447969449952267,0.29977108788901524,0.6305781116048623,0.37928686046418375,4a7f69f8081d48e69854fccbc2babdbe
et_300_none,extratrees,0.29574823135118455,0.6337209302325582,0.3003686944863415,0.29808207145908155,0.6162691639970004,0.4026316802310248,8954b9669eda4cf6bfbb10ab396c050f
et_200_10,extratrees,0.2955321877901944,0.6453488372093024,0.3018664909969258,0.29735562895181716,0.6249774385241014,0.33161111627768425,014cfea50107424297c50b585f7b7ccb
et_500_balanced,extratrees,0.29339339339339343,0.6395348837209303,0.29809121961020696,0.29454310827151803,0.6198512465954327,0.7095632790720248,59281b5f65c147ea99be9cdbc473c2ce
xgb_lr005_md7,xgboost,0.27938697318007666,0.5930232558139535,0.2803497942386831,0.28104150498432806,0.5792141138732959,0.02924152906765558,cdcd6622feee4a58bbc3ce15206fdb52
xgb_lr01_md5,xgboost,0.27548895548895547,0.5872093023255814,0.27386831275720164,0.27875839996149704,0.5749356070286302,0.021967436047160907,fac01704afb344b9885faa781262cc9f
et_100_5,extratrees,0.20982142857142858,0.5755813953488372,0.19163849447825107,0.23185096949242276,0.5210236710963454,0.14480346511803407,7039a48e25a74b50a9c5218f96c990f7


## 8. Aplica el gate sin mirar test

Filtra por F1 macro mínimo; ordena por F1 macro DESC, accuracy DESC y latencia ASC. Conserva
`BEST_RUN_ID`, `BEST_MODEL_URI` y un `selection_report` que diga explícitamente que test no
participó en la selección.

In [0]:
# --- TODO 8: gate y selección del ganador ---
eligible = comparison[comparison["metrics.validation.f1_macro"] >= MIN_VALIDATION_F1_MACRO]
if eligible.empty:
    raise ValueError(f"Ningún candidato superó el gate (F1 macro >= {MIN_VALIDATION_F1_MACRO})")

eligible = eligible.sort_values(
    by=["metrics.validation.f1_macro", "metrics.validation.accuracy", "metrics.validation.latency_ms_per_row"],
    ascending=[False, False, True],
)
best_row = eligible.iloc[0]
BEST_RUN_ID = best_row["run_id"]
BEST_MODEL_URI = best_row.get("tags.candidate.model_uri", f"runs:/{BEST_RUN_ID}/model")
if pd.isna(BEST_MODEL_URI):
    BEST_MODEL_URI = f"runs:/{BEST_RUN_ID}/model"

selection_report = {
    "gate_metric": "validation.f1_macro",
    "gate_threshold": MIN_VALIDATION_F1_MACRO,
    "num_eligible": int(len(eligible)),
    "best_run_id": BEST_RUN_ID,
    "best_model_uri": BEST_MODEL_URI,
    "best_candidate": best_row["tags.candidate_name"],
    "best_f1_macro": float(best_row["metrics.validation.f1_macro"]),
    "best_accuracy": float(best_row["metrics.validation.accuracy"]),
    "best_latency_ms_per_row": float(best_row["metrics.validation.latency_ms_per_row"]),
    "test_participated_in_selection": False,
    "note": "Test set was not used in candidate selection or gate evaluation.",
}
print(f"Gate: {len(eligible)} elegibles (F1 macro >= {MIN_VALIDATION_F1_MACRO})")
print(f"Ganador: {best_row['tags.candidate_name']} | run_id={BEST_RUN_ID}")
print(f"F1 macro={best_row['metrics.validation.f1_macro']:.4f} | Acc={best_row['metrics.validation.accuracy']:.4f}")

Gate: 4 elegibles (F1 macro >= 0.28)
Ganador: et_300_leaf3 | run_id=4a7f69f8081d48e69854fccbc2babdbe
F1 macro=0.2978 | Acc=0.6512


## 9. Carga el modelo desde MLflow, evalúa test una sola vez y completa el run ganador

In [0]:
# --- TODO 9: evaluación final sobre test ---
loaded_model = mlflow.sklearn.load_model(BEST_MODEL_URI)
test_metrics = evaluate_classifier(loaded_model, X_test, y_test)
test_predictions = loaded_model.predict(X_test)

with mlflow.start_run(run_id=BEST_RUN_ID):
    # Log testing dataset
    test_df = X_test.copy()
    test_df["quality_class"] = y_test
    test_dataset = mlflow.data.from_pandas(
        test_df, source=DATASET_PATH, name="wine_test", targets="quality_class"
    )
    mlflow.log_input(test_dataset, context="testing")

    # Log test metrics
    for key, value in test_metrics.items():
        mlflow.log_metric(f"test.{key}", value)

    # Classification report
    test_report = classification_report(y_test, test_predictions, output_dict=True, zero_division=0)
    mlflow.log_dict(test_report, "test_classification_report.json")

    # Tags
    mlflow.set_tag("selection_status", "winner")
    mlflow.set_tag("selection_gate", "validation.f1_macro")
    mlflow.set_tag("selection_threshold", str(MIN_VALIDATION_F1_MACRO))

    # Decision JSON
    selection_report["test_f1_macro"] = float(test_metrics["f1_macro"])
    selection_report["test_accuracy"] = float(test_metrics["accuracy"])
    mlflow.log_dict(selection_report, "selection/decision.json")

print(f"Test F1 macro: {test_metrics['f1_macro']:.4f}")
print(f"Test accuracy: {test_metrics['accuracy']:.4f}")
print(f"Test predictions (first 10): {test_predictions[:10]}")

/local_disk0/.ephemeral_nfs/envs/pythonEnv-cf367e1e-9f5d-4716-9b4b-37fcbe89f8d7/lib/python3.12/site-packages/mlflow/data/dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-cf367e1e-9f5d-4716-9b4b-37fcbe89f8d7/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (flo

Test F1 macro: 0.3304
Test accuracy: 0.6860
Test predictions (first 10): [2 3 3 2 3 3 3 2 2 2]


## 10. Registra el modelo ganador en Unity Catalog

Usa el catálogo y esquema activos, un nombre de tres niveles y tu alias saneado. Añade
descripción y tags. Registra referencia+ganador, asigna `Challenger`, haz smoke test,
promociona, comprueba rollback y deja el ganador como `Champion`. Si recibes `PERMISSION_DENIED`, consulta la
guía: no pegues credenciales ni cambies a recursos de pago.

In [0]:
# --- TODO 10: Model Registry en Unity Catalog ---
mlflow.set_registry_uri("databricks-uc")

# Determine active catalog and schema
try:
    current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
    current_schema = spark.sql("SELECT current_schema()").collect()[0][0]
except Exception:
    current_catalog = "main"
    current_schema = "default"

registered_model_name = f"{current_catalog}.{current_schema}.wine_quality_{STUDENT_ALIAS}"
print(f"Registering model: {registered_model_name}")

# Register the model
registered_version = mlflow.register_model(
    model_uri=BEST_MODEL_URI,
    name=registered_model_name,
    await_registration_for=300,
)
model_version = registered_version.version
print(f"Registered version {model_version}")

registry_client = MlflowClient(registry_uri="databricks-uc")

# Tags
registry_client.set_registered_model_tag(registered_model_name, "student_alias", STUDENT_ALIAS)
registry_client.set_registered_model_tag(registered_model_name, "batch_id", BATCH_ID)
registry_client.set_model_version_tag(registered_model_name, str(model_version), "source_run_id", BEST_RUN_ID)

# Set Challenger alias
registry_client.set_registered_model_alias(
    name=registered_model_name, alias="Challenger", version=model_version,
)
print(f"v{model_version} asignado como Challenger")

# Smoke test with Challenger
challenger_model = mlflow.pyfunc.load_model(f"models:/{registered_model_name}@Challenger")
challenger_preds = challenger_model.predict(X_test.head(3))
print(f"Challenger smoke test: {challenger_preds}")

# Promote to Champion
registry_client.set_registered_model_alias(
    name=registered_model_name, alias="Champion", version=model_version,
)
print(f"v{model_version} promovido a Champion")

# Test rollback
registry_client.delete_registered_model_alias(
    name=registered_model_name, alias="Champion",
)
print("Rollback: Champion eliminado, Challenger sigue activo")

# Re-promote to Champion
registry_client.set_registered_model_alias(
    name=registered_model_name, alias="Champion", version=model_version,
)
champion_version = model_version
print(f"v{model_version} re-promovido a Champion")

Registering model: workspace.default.wine_quality_202114423


Registered model 'workspace.default.wine_quality_202114423' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '2' of model 'workspace.default.wine_quality_202114423': https://dbc-232929c3-e5e6.cloud.databricks.com/explore/data/models/workspace/default/wine_quality_202114423/version/2?o=7474651531910434


Registered version 2
v2 asignado como Challenger


Challenger smoke test: [2 3 3]
v2 promovido a Champion
Rollback: Champion eliminado, Challenger sigue activo
v2 re-promovido a Champion


## 11. Carga `models:/<nombre>@Champion` y realiza un smoke test con tres filas

In [0]:
# --- TODO 11: smoke test del Champion ---
champion_model = mlflow.pyfunc.load_model(f"models:/{registered_model_name}@Champion")
sample_rows = X_test.head(3)
sample_preds = champion_model.predict(sample_rows)
print(f"Modelo cargado desde alias Champion")
print(f"Input: {sample_rows.shape}")
print(f"Predicciones: {sample_preds}")
assert len(sample_preds) == 3, "Debe predecir 3 filas"

Modelo cargado desde alias Champion
Input: (3, 11)
Predicciones: [2 3 3]


## 12. API local con el artefacto Joblib ganador

Resuelve el run de `Champion`, descarga `deployment/model.joblib` y levanta un
`ThreadingHTTPServer` en `127.0.0.1` con puerto automático. Implementa `GET /health` y
`POST /predict`; valida `instances`, columnas ausentes y columnas extra.

In [0]:
# --- TODO 12a: cargar artefacto Joblib del Champion ---
champion_version_info = registry_client.get_model_version_by_alias(
    name=registered_model_name, alias="Champion"
)
champion_run_id = champion_version_info.run_id

joblib_uri = f"runs:/{champion_run_id}/deployment/model.joblib"
local_joblib_path = mlflow.artifacts.download_artifacts(joblib_uri)
serving_model = joblib.load(local_joblib_path)

test_row = X_test.head(1)
test_pred = serving_model.predict(test_row)
print(f"Loaded model.joblib from run {champion_run_id}")
print(f"Prediction for first test row: {test_pred}")
assert len(test_pred) == 1

Loaded model.joblib from run 4a7f69f8081d48e69854fccbc2babdbe
Prediction for first test row: [2]


### 12b. Handler y contrato

Define `PredictionHandler` con `_send_json`, `GET /health` y `POST /predict`. Valida objeto raíz,
lista no vacía de objetos, columnas exactas y errores 400. Instrumenta contadores y latencias.

In [0]:
API_STATS = {"requests_total": 0, "requests_success": 0, "requests_rejected": 0, "requests_error": 0, "latencies_ms": []}
API_STATS_LOCK = threading.Lock()

class PredictionHandler(BaseHTTPRequestHandler):
    def _send_json(self, status_code: int, payload: dict):
        body = json.dumps(payload).encode("utf-8")
        self.send_response(status_code)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        if self.path == "/health":
            with API_STATS_LOCK:
                API_STATS["requests_total"] += 1
                API_STATS["requests_success"] += 1
            self._send_json(200, {"status": "healthy"})
        else:
            with API_STATS_LOCK:
                API_STATS["requests_total"] += 1
                API_STATS["requests_error"] += 1
            self._send_json(404, {"error": "not found"})

    def do_POST(self):
        start = time.perf_counter()
        with API_STATS_LOCK:
            API_STATS["requests_total"] += 1
        try:
            content_length = int(self.headers.get("Content-Length", 0))
            raw = self.rfile.read(content_length)
            try:
                payload = json.loads(raw)
            except (json.JSONDecodeError, ValueError):
                with API_STATS_LOCK:
                    API_STATS["requests_rejected"] += 1
                self._send_json(400, {"error": "invalid JSON"})
                return

            if not isinstance(payload, dict):
                with API_STATS_LOCK:
                    API_STATS["requests_rejected"] += 1
                self._send_json(400, {"error": "root must be an object"})
                return

            instances = payload.get("instances")
            if instances is None:
                with API_STATS_LOCK:
                    API_STATS["requests_rejected"] += 1
                self._send_json(400, {"error": "missing 'instances' field"})
                return

            if not isinstance(instances, list) or len(instances) == 0:
                with API_STATS_LOCK:
                    API_STATS["requests_rejected"] += 1
                self._send_json(400, {"error": "'instances' must be a non-empty list"})
                return

            try:
                input_df = pd.DataFrame(instances)
            except Exception:
                with API_STATS_LOCK:
                    API_STATS["requests_rejected"] += 1
                self._send_json(400, {"error": "could not build DataFrame from instances"})
                return

            missing_cols = [c for c in EXPECTED_FEATURES if c not in input_df.columns]
            extra_cols = [c for c in input_df.columns if c not in EXPECTED_FEATURES]
            if missing_cols or extra_cols:
                with API_STATS_LOCK:
                    API_STATS["requests_rejected"] += 1
                self._send_json(400, {
                    "error": "column mismatch",
                    "missing": missing_cols,
                    "extra": extra_cols,
                })
                return

            predictions = serving_model.predict(input_df[EXPECTED_FEATURES])
            elapsed_ms = (time.perf_counter() - start) * 1000
            with API_STATS_LOCK:
                API_STATS["requests_success"] += 1
                API_STATS["latencies_ms"].append(elapsed_ms)
            self._send_json(200, {
                "predictions": predictions.tolist(),
                "num_instances": len(predictions),
            })
        except Exception as e:
            elapsed_ms = (time.perf_counter() - start) * 1000
            with API_STATS_LOCK:
                API_STATS["requests_error"] += 1
                API_STATS["latencies_ms"].append(elapsed_ms)
            self._send_json(500, {"error": str(e)})

    def log_message(self, format, *args):
        pass

### 12c. Arranque y limpieza idempotente

Arranca en `127.0.0.1` y puerto 0. Define `stop_api()` antes de probar para que un `finally` pueda
cerrar servidor, socket e hilo. No uses `0.0.0.0`.

In [0]:
# --- TODO 12c: arranque y limpieza idempotente ---
api_server = ThreadingHTTPServer(("127.0.0.1", 0), PredictionHandler)
API_PORT = api_server.server_address[1]
api_thread = threading.Thread(target=api_server.serve_forever, daemon=True)
api_thread.start()
API_STOPPED = False

def stop_api():
    global API_STOPPED
    if not API_STOPPED:
        api_server.shutdown()
        api_server.server_close()
        api_thread.join(timeout=5)
        API_STOPPED = True

print(f"API server started on 127.0.0.1:{API_PORT}")

API server started on 127.0.0.1:44361


## 13. Pruebas de contrato y de error

In [0]:
# --- TODO 13: pruebas de contrato y de error ---
def call_json(method: str, path: str, body=None) -> tuple:
    url = f"http://127.0.0.1:{API_PORT}{path}"
    data = json.dumps(body).encode("utf-8") if body is not None else None
    req = Request(url, data=data, method=method)
    if data is not None:
        req.add_header("Content-Type", "application/json")
    try:
        with urlopen(req) as resp:
            return resp.status, json.loads(resp.read())
    except HTTPError as e:
        return e.code, json.loads(e.read())

try:
    # GET /health -> 200
    status, body = call_json("GET", "/health")
    assert status == 200, f"Health check failed: {status}"
    print(f"GET /health -> {status}: {body}")

    # POST valid -> 200
    valid_instances = X_test.head(3).to_dict(orient="records")
    status, body = call_json("POST", "/predict", {"instances": valid_instances})
    assert status == 200, f"Valid prediction failed: {status}"
    print(f"POST /predict (valid) -> {status}: predictions={body['predictions']}")

    # POST only alcohol -> 400
    status, body = call_json("POST", "/predict", {"instances": [{"alcohol": 10.0}]})
    assert status == 400, f"Missing columns should be 400: {status}"
    print(f"POST /predict (only alcohol) -> {status}: {body}")

    # POST non-object JSON -> 400
    status, body = call_json("POST", "/predict", [1, 2, 3])
    assert status == 400, f"Non-object root should be 400: {status}"
    print(f"POST /predict (non-object) -> {status}: {body}")

    print("All API tests passed!")
finally:
    stop_api()

GET /health -> 200: {'status': 'healthy'}
POST /predict (valid) -> 200: predictions=[2, 3, 3]
POST /predict (only alcohol) -> 400: {'error': 'column mismatch', 'missing': ['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar', 'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density', 'pH', 'sulphates'], 'extra': []}
POST /predict (non-object) -> 400: {'error': 'root must be an object'}
All API tests passed!


## 14. Registra un run separado para la evidencia del despliegue

In [0]:
# --- TODO 14: run de observabilidad del deployment ---
with mlflow.start_run(experiment_id=experiment_id, run_name=f"deployment_{BATCH_ID}") as run:
    DEPLOYMENT_RUN_ID = run.info.run_id

    mlflow.set_tag("student_alias", STUDENT_ALIAS)
    mlflow.set_tag("batch_id", BATCH_ID)
    mlflow.set_tag("run_type", "deployment")
    mlflow.set_tag("source_run_id", BEST_RUN_ID)
    mlflow.set_tag("registered_model_name", registered_model_name)
    mlflow.set_tag("registered_model_version", str(champion_version))
    mlflow.set_tag("champion_alias", "Champion")

    # Aggregate stats (no input payloads)
    with API_STATS_LOCK:
        total = API_STATS["requests_total"]
        success = API_STATS["requests_success"]
        rejected = API_STATS["requests_rejected"]
        errors = API_STATS["requests_error"]
        latencies = list(API_STATS["latencies_ms"])

    mlflow.log_metric("requests_total", total)
    mlflow.log_metric("requests_success", success)
    mlflow.log_metric("requests_rejected", rejected)
    mlflow.log_metric("requests_error", errors)
    if latencies:
        mlflow.log_metric("latency_avg_ms", sum(latencies) / len(latencies))
        mlflow.log_metric("latency_max_ms", max(latencies))
        mlflow.log_metric("latency_min_ms", min(latencies))

    evidence = {
        "deployment_run_id": DEPLOYMENT_RUN_ID,
        "source_run_id": BEST_RUN_ID,
        "registered_model": registered_model_name,
        "registered_model_version": champion_version,
        "api_port": API_PORT,
        "requests_total": total,
        "requests_success": success,
        "requests_rejected": rejected,
        "requests_error": errors,
        "latencies_ms": latencies,
    }
    mlflow.log_dict(evidence, "deployment/evidence.json")

print(f"Deployment run: {DEPLOYMENT_RUN_ID}")
print(f"Total: {total} | Success: {success} | Rejected: {rejected} | Errors: {errors}")

2026/10/01 10:56:33 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: An error occurred while calling o566.extraContext. Trace:
py4j.security.Py4JSecurityException: Method public scala.collection.immutable.Map com.databricks.backend.common.rpc.CommandContext.extraContext() is not whitelisted on class class com.databricks.backend.common.rpc.CommandContext
	at py4j.security.WhitelistingPy4JSecurityManager.checkCall(WhitelistingPy4JSecurityManager.java:473)
	at py4j.Gateway.invoke(Gateway.java:305)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.$anonfun$invokeMethod$1(InstrumentedCallCommand.scala:19)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke(InstrumentedPy4jCommand.scala:28)
	at com.databricks.backend.daemon.driver.InstrumentedPy4jCommand.instrumentedInvoke$(InstrumentedPy4jCommand.scala:18)
	at com.databricks.backend.daemon.driver.InstrumentedCallCommand.instrumentedInvoke(Instrumente

Deployment run: 00d1c5724d1b47efb0af1b35e790ba7c
Total: 4 | Success: 2 | Rejected: 2 | Errors: 0


## 15. Apaga la API y entrega el resumen del ciclo

In [0]:
# --- TODO 15: cierre y resumen ---
if not API_STOPPED:
    stop_api()
assert API_STOPPED, "API server should be stopped"

print("=" * 60)
print("RESUMEN DEL CICLO DE VIDA")
print("=" * 60)
print(f"Experiment ID:   {experiment_id}")
print(f"Batch ID:        {BATCH_ID}")
print(f"Best run ID:     {BEST_RUN_ID}")
print(f"Registered model:{registered_model_name}")
print(f"Champion version: {champion_version}")
print(f"Champion alias:  Champion")
print(f"Deployment run:  {DEPLOYMENT_RUN_ID}")
print("=" * 60)

RESUMEN DEL CICLO DE VIDA
Experiment ID:   96587636299818
Batch ID:        8b01ae64
Best run ID:     4a7f69f8081d48e69854fccbc2babdbe
Registered model:workspace.default.wine_quality_202114423
Champion version: 2
Champion alias:  Champion
Deployment run:  00d1c5724d1b47efb0af1b35e790ba7c
